# SAFER FDS — AI Engine Model V3 Training Pipeline

Notebook ini digunakan untuk melatih model **XGBoost + LightGBM Ensemble (Model V3)** di Kaggle menggunakan dataset transaksi FDS yang sudah di-generate sebelumnya. Notebook ini melakukan **Advanced Feature Engineering** tingkat industri dan mengevaluasi performa model secara detail per skenario fraud spesifik perbankan Indonesia.

In [ ]:
# 1. Install Dependencies di Lingkungan Kaggle
!pip install xgboost lightgbm scikit-learn pandas numpy joblib

In [ ]:
# 2. Import Libraries
import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# ML libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, accuracy_score
)
import xgboost as xgb
import lightgbm as lgb

## 3. Dataset Path Configuration
Upload file `train_transactions.csv` dan `test_transactions.csv` sebagai dataset baru di Kaggle, kemudian sesuaikan path direktori input di bawah ini sesuai nama dataset yang kamu buat.

In [ ]:
# Tentukan path dataset input di Kaggle
# Ganti 'nama-dataset-kamu' sesuai folder input di Kaggle sidebar
DATASET_DIR = Path("/kaggle/input/safer-fds-v3-dataset")

TRAIN_PATH = DATASET_DIR / "train_transactions.csv"
TEST_PATH = DATASET_DIR / "test_transactions.csv"

# Cek ketersediaan file
if not TRAIN_PATH.exists():
    # Fallback pencarian file otomatis jika nama folder berbeda
    print("Mencari lokasi dataset...")
    input_dirs = list(Path("/kaggle/input").glob("**/*transactions.csv"))
    if input_dirs:
        DATASET_DIR = input_dirs[0].parent
        TRAIN_PATH = DATASET_DIR / "train_transactions.csv"
        TEST_PATH = DATASET_DIR / "test_transactions.csv"
        print(f"Dataset ditemukan di: {DATASET_DIR}")
    else:
        print("ERROR: File dataset tidak ditemukan! Pastikan Anda sudah menambahkan dataset ke notebook.")

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")

In [ ]:
# 4. Load Dataset
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Data Training Loaded: {len(train_df):,} baris ({train_df['is_fraud'].sum():,} fraud)")
print(f"Data Testing Loaded: {len(test_df):,} baris ({test_df['is_fraud'].sum():,} fraud)")

## 5. Advanced Feature Engineering
Menambahkan fitur jam siklikal (sin/cos) dan metrik rasio penipuan.

In [ ]:
def add_advanced_features(df):
    timestamps = pd.to_datetime(df["timestamp"])
    hours = timestamps.dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * hours / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * hours / 24.0)
    df["amount_to_age_ratio"] = df["amount"] / (df["account_age_days"] + 1.0)
    df["dist_to_velocity_ratio"] = df["geo_distance_km"] / (df["velocity_count"].astype(float) + 1.0)
    df["amount_to_distance_ratio"] = df["amount"] / (df["geo_distance_km"] + 1.0)
    return df

## 6. Preprocessing (Label Encoding & Scaling)

In [ ]:
MODEL_FEATURES = [
    "sender_bank", "sender_lat", "sender_lng", "receiver_bank", "receiver_lat", "receiver_lng",
    "amount", "payment_rail", "ewallet_provider", "merchant", "merchant_category", "channel",
    "device_type", "device_brand", "is_new_device", "account_age_days", "is_velocity_anomaly",
    "is_geo_mismatch", "is_off_hours", "is_high_value_for_rail", "is_suspicious_ip",
    "is_risky_merchant", "is_new_account", "has_failed_attempts", "is_device_mismatch",
    "is_sim_swap", "is_unusual_beneficiary", "velocity_count", "geo_distance_km",
    "hour_sin", "hour_cos", "amount_to_age_ratio", "dist_to_velocity_ratio", "amount_to_distance_ratio"
]

CATEGORICAL_COLS = [
    "sender_bank", "receiver_bank", "payment_rail", "ewallet_provider",
    "merchant", "merchant_category", "channel", "device_type", "device_brand"
]

SCALED_COLS = [
    "amount", "account_age_days", "velocity_count", "geo_distance_km",
    "hour_sin", "hour_cos", "amount_to_age_ratio", "dist_to_velocity_ratio", "amount_to_distance_ratio"
]

# Jalankan Feature Engineering
train_df = add_advanced_features(train_df)
test_df = add_advanced_features(test_df)

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df["is_fraud"].copy()
X_test = test_df[MODEL_FEATURES].copy()
y_test = test_df["is_fraud"].copy()

# Encode categoricals
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X_train[col] = X_train[col].fillna("None").astype(str)
    X_test[col] = X_test[col].fillna("None").astype(str)
    all_val = pd.concat([X_train[col], X_test[col]]).unique()
    le.fit(all_val)
    X_train[col] = le.transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le

# Scale numerics
scaler = StandardScaler()
X_train[SCALED_COLS] = scaler.fit_transform(X_train[SCALED_COLS])
X_test[SCALED_COLS] = scaler.transform(X_test[SCALED_COLS])

## 7. Melatih Model V3 Ensemble (XGBoost + LightGBM)

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

print("Training LightGBM...")
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.05,
    "n_estimators": 300,
    "is_unbalance": True,
    "random_state": 42,
    "verbose": -1
}
lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=300)

## 8. Evaluasi Model V3 (Metrik Industri & Skenario Breakdown)

In [ ]:
# Probs & preds
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
lgb_probs = lgb_model.predict(X_test.to_numpy())
ens_probs = (xgb_probs + lgb_probs) / 2.0
ens_preds = (ens_probs >= 0.5).astype(int)

# Ensemble metrics
precision, recall, f1, _ = precision_recall_fscore_support(y_test, ens_preds, average="binary")
roc_auc = roc_auc_score(y_test, ens_probs)
pr_auc = average_precision_score(y_test, ens_probs)
acc = accuracy_score(y_test, ens_preds)

print("=== ENSEMBLE EVALUATION GLOBAL METRICS ===")
print(f"Accuracy:  {acc:.4%}")
print(f"Precision: {precision:.4%}")
print(f"Recall:    {recall:.4%}")
print(f"F1 Score:  {f1:.4%}")
print(f"ROC-AUC:   {roc_auc:.4%}")
print(f"PR-AUC:    {pr_auc:.4%}")

# Pattern Specific Breakdown
print("\n=== FRAUD SCENARIO EVALUATION BREAKDOWN ===")
test_patterns = test_df["fraud_pattern"].copy()
unique_patterns = [p for p in test_patterns.unique() if p not in ["Normal", "General Fraud"]]

for pattern in unique_patterns:
    idx = (test_patterns == "Normal") | (test_patterns == pattern)
    y_sub = y_test[idx]
    preds_sub = ens_preds[idx]
    prec_sub, rec_sub, f1_sub, _ = precision_recall_fscore_support(y_sub, preds_sub, average="binary", zero_division=0)
    print(f"{pattern} -> Precision: {prec_sub:.4%}, Recall: {rec_sub:.4%}, F1: {f1_sub:.4%}")

## 9. Export Artifacts V3
Menyimpan bobot model ke file keluaran (/kaggle/working/)

In [ ]:
xgb_model.save_model("xgb_model_v3.json")
lgb_model.save_model("lgb_model_v3.txt")
joblib.dump(label_encoders, "label_encoders_v3.pkl")
joblib.dump(scaler, "scaler_v3.pkl")

print("Bobot Model V3 sukses disimpan di output directory!")